# Welcome to Mechanex

**Mechanex** allows you to optimize, align, and correct your generative AI models in production. It acts as a **runtime control layer** for LLMs, enabling you to improve model behavior at inference time through **policies**, without the need for retraining or fine-tuning.

In this notebook, we'll walk through the core features of the library:
1. **Execution Modes**: Local, Remote, and Hybrid.
2. **Advanced Sampling**: Beyond greedy, exploring Top-K, Top-P, and more.
3. **Structured Output**: Enforcing JSON schemas at runtime.
4. **Steering Vectors**: Controlling model behavior through activation engineering.
5. **SAE Behaviors**: Behavioral detection and alignment using Sparse Autoencoders.
6. **Policy Evaluation**: Comparing and testing different configurations.

## 1. Setup and Initialization

First, we'll install Mechanex and initialize the client. To use the remote features, you'll need an API key.

In [ ]:
!pip install mechanex

import mechanex as mx
import os
import json

print(f"Mechanex version: {mx.__version__ if hasattr(mx, '__version__') else 'Installed'}")

### Authentication

If you have an API key, you can set it here. If you're using Mechanex locally for the first time, you can also run `!mechanex signup` in a terminal.

In [ ]:
# mx.set_key("your-api-key-here")
# mx.set_execution_mode("remote") # or "local", or "auto"

## 2. Local vs. Remote Execution

Mechanex is uniquely designed to run the same policies on your local machine and on hosted infrastructure.

In [ ]:
# Load a small model locally (requires transformer-lens)
try:
    mx.load_model("gpt2-small")
    mx.set_execution_mode("local")
    print("Model loaded locally.")
except Exception as e:
    print(f"Local load skipped or failed: {e}. Defaulting to remote auto-mode.")
    mx.set_execution_mode("auto")

## 3. Advanced Sampling Strategies

Mechanex supports a wide variety of sampling methods to control the creativity and determinism of the model.

In [ ]:
prompt = "The key to building reliable AI systems is"

sampling_methods = ["greedy", "top-k", "top-p", "min-p", "typical"]

for method in sampling_methods:
    output = mx.generation.generate(
        prompt=prompt,
        sampling_method=method,
        max_tokens=20
    )
    print(f"--- Method: {method} ---\n{output}\n")

## 4. Structured Output with Policies

One of the most powerful features of Mechanex is the ability to enforce JSON schemas at runtime without complicated prompt engineering. 

In [ ]:
target_schema = {
    "type": "object",
    "required": ["ticket_id", "priority", "summary"],
    "properties": {
        "ticket_id": {"type": "string"},
        "priority": {"type": "string", "enum": ["low", "medium", "high"]},
        "summary": {"type": "string"},
    },
}

# Create a policy preset
policy = mx.policy.strict_json_extraction(schema=target_schema, name="support_json_v1")

# Run the policy
res = mx.policy.run(
    prompt="Extract a support ticket from: The user reports their login is failing. Urgent. Ticket number T-202.",
    policy=policy,
    include_trace=True
)

print("Resulting JSON:")
print(json.dumps(json.loads(res["output"]), indent=2))
print(f"\nPolicy Accepted: {res['accepted']}")

## 5. Steering Vectors (Activation Engineering)

Instead of fine-tuning, you can 'steer' the model by injecting activation patterns derived from small sets of contrastive examples.

In [ ]:
# Sample contrastive dataset for steering toward security-conscious responses
steering_data = [
    {"prompt": "User: Summarize this report.\nAssistant:", "positive_answer": "Key findings: patch exposed endpoints.", "negative_answer": "It's too long."},
    {"prompt": "User: Give hardening tips.\nAssistant:", "positive_answer": "Use auth scopes, strict validation.", "negative_answer": "Just keep coding."}
]

prompts = [x["prompt"] for x in steering_data]
positive = [x["positive_answer"] for x in steering_data]
negative = [x["negative_answer"] for x in steering_data]

# 1. Generate the vector
vector_id = mx.steering.generate_vectors(
    prompts=prompts,
    positive_answers=positive,
    negative_answers=negative,
    method="caa"
)
print(f"Generated Vector ID: {vector_id}")

# 2. Apply the vector during generation
steered_output = mx.generation.generate(
    prompt="How should we design a secure API, briefly?",
    steering_vector=vector_id,
    steering_strength=1.5,
    max_tokens=50
)

print("\nSteered Output:")
print(steered_output)

## 6. SAE Behaviors (Alignment & Correction)

Mechanex uses Sparse Autoencoders (SAE) to detect and correct specific model behaviors in real-time.

In [ ]:
# Create an SAE-based behavior for conciseness
sae_behavior_data = [
    {"prompt": "Explain this feature.", "positive_answer": "Short, direct summary.", "negative_answer": "Long repetitive narrative."},
    {"prompt": "Password reset help.", "positive_answer": "Steps in 3 lines.", "negative_answer": "Overly verbose monologue."}
]

behavior = mx.sae.create_behavior(
    behavior_name="concise_mode",
    prompts=[x["prompt"] for x in sae_behavior_data],
    positive_answers=[x["positive_answer"] for x in sae_behavior_data],
    negative_answers=[x["negative_answer"] for x in sae_behavior_data],
    description="Encourage concise style"
)

# Generate with behavior correction enabled
corrected_output = mx.sae.generate(
    prompt="Describe how to optimize model inference in 3 points.",
    behavior_names=["concise_mode"],
    max_new_tokens=60
)

print("Output with Conciseness Alignment:")
print(corrected_output)

## 7. Policy Evaluation & Comparison

You can easily test and compare different policies across multiple prompts to see which one performs better for your specific task.

In [ ]:
# Compare two different policies
policy_a = mx.policy.fast_tool_router()
policy_b = policy # Our previously defined JSON extraction policy

comparison = mx.policy.compare(
    prompt="Extract order_id and status from: Order #55 is being shipped.",
    policies=[policy_a, policy_b]
)

print("Comparison Results:")
for i, res in enumerate(comparison):
    print(f"Policy {i} Output: {res['output']}")
    print(f"Policy {i} Accepted: {res['accepted']}\n")

## 8. Deployment & Serving

Finally, Mechanex can start an OpenAI-compatible server that automatically applies your policies and behaviors.

In [ ]:
# Start serving locally on port 8001
# Note: Running this will block the loop in a normal notebook cell.
# mx.serve(port=8001, corrected_behaviors=["toxicity"])

### Summary
You've explored the core capabilities of Mechanex:
- **Remote execution** for all inference time model optimization.
- **Fine-grained control** over sampling and structured output.
- **State-of-the-art alignment** via Steering Vectors and SAE Behaviors.
- **Rapid iteration** with policy comparison and evaluation.

For more, visit our [documentation](https://docs.axioniclabs.ai/products/mechanex/introduction) or check out the [examples/](examples/) directory.